# HOUSING WORKBOOK

### DATASET LINK  

https://www.kaggle.com/datasets/vallabhadattap/kingcountyhousing

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import stats

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score



### 1. Get and Understand the data

In [ ]:
df = pd.read_csv("KC_housing_data.csv")
print(df.shape)
print(df.head)

### 2. Explore and Clean

In [ ]:
missing = df.isna().sum() #missing values in the dataset check
missing

In [ ]:
df["street"].describe()

In [ ]:
dups = df[df["street"].duplicated(keep=False)].sort_values(["street", "date"]) # duplicate houses on the dataset were houses that were sold multiple times
dups[["street", "date", "price", "sqft_living", "bedrooms"]].head(20)

In [ ]:
print("duplicate rows :" , df.duplicated().sum())

In [ ]:
df = df[df["price"] > 0] # no NaNs but some values can still be 0 which would skew the result

In [ ]:
(df["price"] == 0).sum()

# 3.Choosing X AND Y variables

In [ ]:
features = ["bedrooms", "bathrooms", "sqft_living", "sqft_lot", "floors", "condition"]
price = "price"

X = df[features]
Y = df[price]

In [ ]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size = 0.2, random_state = 50)

# 5. Check initial regression

# 6. Create and train the model

In [ ]:
model = LinearRegression()    # The negative bedrooms coefficient looks wrong but has a standard explanation. It’s the effect of adding a bedroom while holding sqft_living fixed. If total floor area doesn’t change, an extra bedroom means smaller rooms, which buyers value less. Bedrooms and living area are strongly correlated, so the model splits their effect in a way that looks odd. This is multicollinearity
model.fit(X_train, Y_train)

print("Intercept:", model.intercept_)
pd.Series(model.coef_, index=features).sort_values()

# 7. Make Predictions

In [ ]:
Y_pred = model.predict(X_test)
pd.DataFrame({"actual": Y_test, "predicted": Y_pred})

In [ ]:
plt.figure(figsize=(7,6))
plt.scatter(Y_test, Y_pred, s=8, alpha = 0.4)


lims = [min(Y_test.min(), Y_pred.min()), max(Y_test.max(), Y_pred.max())]
plt.plot(lims, lims, "r--", label="Perfect prediction")
plt.xlim(0, 2_000_000)
plt.ylim(0, 2_000_000)
plt.xlabel("Actual price")
plt.ylabel("Predicted price")
plt.title("Actual vs predicted house prices")

In [ ]:
check = X_test.copy()
check["actual"] = Y_test
check["predicted"] = Y_pred
check["error"] = check["actual"] - check["predicted"]
check = check.join(df[["city", "statezip"]])
check.sort_values("error", ascending=False).head(10)

In [ ]:
residuals = Y_test - Y_pred
plt.scatter(Y_pred, residuals, s=6, alpha=0.4)
plt.axhline(0, color="r", linestyle="--")
plt.xlabel("Predicted price"); plt.ylabel("Residual"); plt.show()

In [ ]:
from scipy import stats
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].hist(residuals, bins=50); ax[0].set_title("Residual distribution")
stats.probplot(residuals, plot=ax[1]); ax[1].set_title("Q-Q plot")
plt.show()

In [ ]:
fig, ax = plt.subplots(2, 3, figsize=(15, 8))
for a, col in zip(ax.ravel(), features):
    a.scatter(X_test[col], residuals, s=5, alpha=0.3)
    a.axhline(0, color="r"); a.set_title(col)
plt.show()

In [ ]:
plt.hist(Y_test, bins=50, range=(0, 2_000_000), alpha=0.5, label="actual")
plt.hist(Y_pred, bins=50, range=(0, 2_000_000), alpha=0.5, label="predicted")
plt.legend();
plt.title("Variation in Actual vs Predicted price");
plt.xlabel("Price"); 
plt.show() 